# Step 1 — Must-preserve extraction, LLM only, one row per prompt (HLQC + PAIR, 20 prompts each)

- **HLQC:** high-quality sessions, client (`C:`) turns only. **PAIR:** the `prompt` column.
- From each dataset, **20 prompts closest to the median length** (or mean / longest — `SELECT_BY`).
- The LLM reads each prompt once and returns (1) all must-preserve **spans** and (2) the **12 attributes for the whole prompt** (not per span).
- Output: `outputs/hlqc_mustpreserve.csv` and `outputs/pair_mustpreserve.csv` — **one row per prompt**:

`prompt` → `spans` (all spans in one column, separated by ` | `) → 12 attributes → `core_meaning`

| # | Attribute (whole prompt) | Values |
|---|---|---|
| 1 | `category` | all present: symptom, health_condition, emotion, life_event, target_behavior, risk, quantity |
| 2 | `label` | main topic, 2–5 words |
| 3 | `intensity` | low / moderate / high / unspecified |
| 4 | `severity` | low / moderate / high |
| 5 | `risk_type` | all present (incl. denied), or not_risk |
| 6 | `negated` | True if any key meaning is negated |
| 7 | `experiencer` | everyone described: client, partner, child, parent, … |
| 8 | `temporality` | current / past / future / habitual / mixed / unspecified |
| 9 | `hypothetical` | True / False |
| 10 | `certainty` | certain / hedged / mixed |
| 11 | `mi_talk` | change / sustain / ambivalent / neutral |
| 12 | `mi_talk_evidence` | verbatim span |

Lists inside a cell are separated by `; `. `prompt_char_len`, `prompt_word_len` are descriptive stats only.

**Run order:** top to bottom. LLM results are cached in `outputs/llm_cache.jsonl`, so re-running never pays twice.

## 1. Install

In [ ]:
!pip -q install openai pandas tqdm

## 2. Config — edit here only
Colab: upload `HighLowQualityCounseling.zip` and `pair_data_eng.csv` to `/content` (Files panel). Locally: set the paths. Remove a line from `DATASETS` to skip that dataset.

In [ ]:
DATASETS = {                                   # name -> path (.zip/folder = HLQC, .csv = PAIR)
    "hlqc": "/content/HighLowQualityCounseling.zip",
    "pair": "/content/pair_data_eng.csv",
}
OUT_DIR            = "outputs"
QUALITY            = "high"          # HLQC only: "high" or "low"
N_PROMPTS          = 20              # per dataset
SELECT_BY          = "median"        # "median", "mean" or "longest"
LENGTH_BY          = "chars"         # "chars" or "words"
MAX_PER_TRANSCRIPT = 0               # 0 = no cap; 1 = at most one prompt per session
DEDUP              = True            # drop near-duplicate turns (duplicate sessions in corpus)
MERGE_CONSECUTIVE  = True            # join consecutive C: lines into one turn

MODEL              = "gpt-4o-mini"

## 3. Load data and select the prompts

In [ ]:
import os, re, csv, io, json, zipfile, hashlib
from difflib import SequenceMatcher
import pandas as pd
from tqdm.auto import tqdm
os.makedirs(OUT_DIR, exist_ok=True)

TURN_RE = re.compile(r"^\s*([TC])\s*[:;]\s*(.*)$")  # "T;" typo exists in high_104


def norm(t) -> str:
    return (str(t).replace("\ufeff", "").replace("\u2019", "'").replace("\u2018", "'")
            .replace("\u201c", '"').replace("\u201d", '"'))


def _labels_from_csv(fh) -> dict:
    return {r["id"].strip(): r["label"].strip() for r in csv.DictReader(fh)}


def read_dataset(path: str) -> tuple[dict, dict]:
    """Return ({transcript_id: raw_text}, {transcript_id: label}). `path` = the .zip or any folder containing it."""
    files, labels = {}, {}
    if path.lower().endswith(".zip"):
        with zipfile.ZipFile(path) as z:
            names = [n for n in z.namelist() if "__MACOSX" not in n and not n.endswith("/")]
            for n in names:
                if n.endswith("labels.csv"):
                    labels = _labels_from_csv(io.TextIOWrapper(z.open(n), encoding="utf-8-sig"))
                elif "/transcripts/" in n:
                    files[os.path.basename(n)] = z.read(n).decode("utf-8-sig", errors="replace")
    else:
        for root, _, fnames in os.walk(path):
            if "__MACOSX" in root:
                continue
            for f in fnames:
                p = os.path.join(root, f)
                if f == "labels.csv":
                    with open(p, encoding="utf-8-sig") as fh:
                        labels = _labels_from_csv(fh)
                elif os.path.basename(root) == "transcripts" and not f.startswith("."):
                    with open(p, encoding="utf-8-sig", errors="replace") as fh:
                        files[f] = fh.read()
    if not files:
        raise FileNotFoundError(f"No transcripts found under {path}")
    return files, labels


def parse_turns(text: str) -> list[tuple[str, str]]:
    """[(speaker, text)]; lines without a speaker tag are appended to the previous turn."""
    turns: list[list[str]] = []
    for line in norm(text).splitlines():
        if not line.strip():
            continue
        m = TURN_RE.match(line)
        if m:
            turns.append([m.group(1), m.group(2).strip()])
        elif turns:
            turns[-1][1] += " " + line.strip()
    return [(s, " ".join(t.split())) for s, t in turns]


def client_utterances(files: dict, labels: dict, quality: str = "high",
                      merge_consecutive: bool = True) -> pd.DataFrame:
    rows = []
    for tid in sorted(files):
        label = labels.get(tid) or tid.split("_")[0]
        if label != quality:
            continue
        turns = parse_turns(files[tid])
        if merge_consecutive:  # one client turn split over several C: lines -> one utterance
            merged = []
            for s, t in turns:
                if merged and s == "C" and merged[-1][0] == "C":
                    merged[-1] = ("C", merged[-1][1] + " " + t)
                else:
                    merged.append((s, t))
            turns = merged
        c_idx = 0
        for turn_idx, (s, t) in enumerate(turns):
            if s == "C" and t:
                rows.append({"transcript_id": tid, "turn_index": turn_idx,
                             "client_turn_index": c_idx, "prompt": t})
                c_idx += 1
    df = pd.DataFrame(rows)
    df["prompt_char_len"] = df["prompt"].str.len()
    df["prompt_word_len"] = df["prompt"].str.split().str.len()
    return df


def _near_dup(a: str, b: str, thr: float) -> bool:
    wa, wb = a.lower().split(), b.lower().split()
    sm = SequenceMatcher(None, wa, wb, autojunk=False)
    return sm.real_quick_ratio() >= thr and sm.quick_ratio() >= thr and sm.ratio() >= thr


def load_pair(path: str) -> pd.DataFrame:
    """PAIR csv: one client prompt per row (reflection columns hq1..lq5 are kept)."""
    df = pd.read_csv(path)
    df["prompt"] = df["prompt"].map(lambda t: " ".join(norm(t).split()))
    df.insert(0, "transcript_id", [f"pair_{i:03d}" for i in range(len(df))])
    df.insert(1, "turn_index", range(len(df)))
    df.insert(2, "client_turn_index", 0)
    df["prompt_char_len"] = df["prompt"].str.len()
    df["prompt_word_len"] = df["prompt"].str.split().str.len()
    return df


def select_prompts(df: pd.DataFrame, n: int = 20, length_by: str = "chars", select_by: str = "median",
                   dedup: bool = True, dedup_threshold: float = 0.8, max_per_transcript: int = 0) -> pd.DataFrame:
    """N prompts closest to the median / mean length (or the N longest). Dedup drops near-identical
    turns (the HLQC corpus has duplicate sessions, e.g. high_011 == high_014, high_017 ~ high_154)."""
    key = "prompt_char_len" if length_by == "chars" else "prompt_word_len"
    df = df.copy()
    if select_by == "longest":
        target = None
        df["_rank"] = -df[key]
    else:
        target = df[key].median() if select_by == "median" else df[key].mean()
        df["_rank"] = (df[key] - target).abs()
    ordered = df.sort_values(["_rank", "transcript_id", "turn_index"])
    picked, per_tid = [], {}
    for _, r in ordered.iterrows():
        if max_per_transcript and per_tid.get(r.transcript_id, 0) >= max_per_transcript:
            continue
        if dedup and any(_near_dup(r.prompt, p.prompt, dedup_threshold) for p in picked):
            continue
        picked.append(r)
        per_tid[r.transcript_id] = per_tid.get(r.transcript_id, 0) + 1
        if len(picked) == n:
            break
    out = pd.DataFrame(picked).drop(columns="_rank").reset_index(drop=True)
    out.insert(0, "utt_id", range(len(out)))
    out["length_target"] = target
    return out


In [ ]:
sels = {}
for name, path in DATASETS.items():
    if path.lower().endswith(".csv"):
        utts = load_pair(path)
    else:
        files, labels = read_dataset(path)
        utts = client_utterances(files, labels, QUALITY, merge_consecutive=MERGE_CONSECUTIVE)
    sel = select_prompts(utts, N_PROMPTS, LENGTH_BY, SELECT_BY, dedup=DEDUP, max_per_transcript=MAX_PER_TRANSCRIPT)
    sels[name] = sel
    key = "prompt_char_len" if LENGTH_BY == "chars" else "prompt_word_len"
    print(f"{name}: {len(utts)} client utterances | {SELECT_BY} {LENGTH_BY} = {sel.length_target[0]} | "
          f"selected {len(sel)} prompts, {LENGTH_BY} {sel[key].min()}-{sel[key].max()}")

for name, sel in sels.items():
    display(sel[["utt_id", "transcript_id", "prompt_char_len", "prompt_word_len", "prompt"]].assign(dataset=name))

## 4. LLM setup
Key lookup order: Colab Secrets (`OPENAI_API_KEY`, key icon in the left sidebar) → environment variable → prompt. Never paste it into code.

In [ ]:
from getpass import getpass
if not os.getenv("OPENAI_API_KEY"):
    try:
        from google.colab import userdata
        os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
    except Exception:
        os.environ["OPENAI_API_KEY"] = getpass("Paste OpenAI key: ")


SYS = """You annotate CLIENT utterances from health-behavior / motivational-interviewing (MI) counseling.
The text is an automatic speech transcript: no punctuation, fillers, repetitions and recognition errors.
The goal: find everything a Spanish culture-adapted translation MUST NOT lose or distort.

1) spans: list every must-preserve piece of meaning, each copied VERBATIM from the text (short, <= 12 words),
   even if it is ungrammatical; never correct or paraphrase. One span per distinct piece of meaning.
   Do not extract style, fillers, or politeness. Cover symptoms, health conditions, emotions, life events /
   stressors, target behaviors, risks (including DENIED risk such as "I'm not suicidal"), and quantities
   (amounts, frequencies, durations, ages, probabilities).

2) Then label the WHOLE utterance (not each span):
- category: every category present: symptom, health_condition, emotion, life_event, target_behavior
  (smoking, alcohol, diet, exercise, medication, testing, sex, ...), risk, quantity.
- label: the main topic in 2-5 words (e.g. "smoking cessation", "medication adherence").
- intensity: low / moderate / high from intensifiers and wording; unspecified if no cue.
- severity: overall clinical/emotional severity of the utterance: low / moderate / high.
- risk_type: every risk type present, including denied ones (suicidal_ideation, self_harm, harm_to_others,
  substance_misuse, risky_sexual, impaired_driving, treatment_refusal, fatalism, other); ["not_risk"] if none.
- negated: true if any must-preserve meaning is stated as NOT happening / not true
  (losing the negation in translation would flip the meaning).
- experiencer: everyone whose symptom/behavior/emotion is described ("my husband still smokes" -> partner).
- temporality: current, past ("I used to bike", "I quit"), future, habitual, mixed (clearly more than one),
  or unspecified.
- hypothetical: true if the utterance contains a conditional/imagined situation ("If I sign up to be a donor...").
- certainty: certain; hedged if qualified by maybe / probably / I'm not sure / I guess / sometimes-type hedges;
  mixed if some statements are hedged and others are not.
- mi_talk: change (desire/ability/reason/need/commitment to change), sustain (arguments against change),
  ambivalent (both), neutral. mi_talk_evidence: the verbatim span that best shows it ("" if neutral).
- core_meaning: one sentence a translation must convey."""

CATEGORIES = ["symptom", "health_condition", "emotion", "life_event", "target_behavior", "risk", "quantity"]
RISK_TYPES = ["not_risk", "suicidal_ideation", "self_harm", "harm_to_others", "substance_misuse",
              "risky_sexual", "impaired_driving", "treatment_refusal", "fatalism", "other"]
EXPERIENCERS = ["client", "partner", "child", "parent", "other_family", "friend_peer", "other_person", "general"]
def enum(vals): return {"type": "string", "enum": vals}
def enum_list(vals): return {"type": "array", "items": enum(vals)}

SCHEMA = {"name": "must_preserve_prompt", "strict": True, "schema": {
  "type": "object", "additionalProperties": False,
  "required": ["spans", "category", "label", "intensity", "severity", "risk_type", "negated", "experiencer",
               "temporality", "hypothetical", "certainty", "mi_talk", "mi_talk_evidence", "core_meaning"],
  "properties": {
    "spans": {"type": "array", "items": {"type": "string"}},
    "category": enum_list(CATEGORIES),
    "label": {"type": "string"},
    "intensity": enum(["low", "moderate", "high", "unspecified"]),
    "severity": enum(["low", "moderate", "high"]),
    "risk_type": enum_list(RISK_TYPES),
    "negated": {"type": "boolean"},
    "experiencer": enum_list(EXPERIENCERS),
    "temporality": enum(["current", "past", "future", "habitual", "mixed", "unspecified"]),
    "hypothetical": {"type": "boolean"},
    "certainty": enum(["certain", "hedged", "mixed"]),
    "mi_talk": enum(["change", "sustain", "ambivalent", "neutral"]),
    "mi_talk_evidence": {"type": "string"},
    "core_meaning": {"type": "string"}}}}


ASR_LINE = "The text is an automatic speech transcript: no punctuation, fillers, repetitions and recognition errors.\n"
SYS_PAIR = SYS.replace(ASR_LINE, "")          # PAIR prompts are written text, not ASR
def sys_for(path): return SYS_PAIR if path.lower().endswith(".csv") else SYS


class LLMExtractor:
    def __init__(self, model: str = "gpt-4o-mini", cache_path: str = "outputs/llm_cache.jsonl",
                 base_url: str | None = None, retries: int = 3):
        from openai import OpenAI
        self.client = OpenAI(base_url=base_url) if base_url else OpenAI()
        self.model, self.cache_path, self.retries = model, cache_path, retries
        self.cache = self._load_cache()

    def _key(self, text: str, sys_prompt: str) -> str:  # cache invalidates if model, prompt or text change
        return hashlib.sha1(f"{self.model}\n{sys_prompt}\n{text}".encode()).hexdigest()

    def _load_cache(self) -> dict:
        out = {}
        if os.path.exists(self.cache_path):
            with open(self.cache_path, encoding="utf-8") as f:
                for line in f:
                    r = json.loads(line)
                    if "error" not in r["out"]:
                        out[r["key"]] = r
        return out

    def extract(self, text: str, sys_prompt: str = SYS) -> dict:
        text = norm(text)
        k = self._key(text, sys_prompt)
        if k in self.cache:
            return self.cache[k]
        err = None
        for _ in range(self.retries):
            try:
                r = self.client.chat.completions.create(
                    model=self.model, temperature=0, seed=42,
                    response_format={"type": "json_schema", "json_schema": SCHEMA},
                    messages=[{"role": "system", "content": sys_prompt}, {"role": "user", "content": text}])
                rec = {"key": k, "model": self.model, "out": json.loads(r.choices[0].message.content),
                       "in_tok": r.usage.prompt_tokens, "out_tok": r.usage.completion_tokens}
                os.makedirs(os.path.dirname(self.cache_path) or ".", exist_ok=True)
                with open(self.cache_path, "a", encoding="utf-8") as f:
                    f.write(json.dumps(rec, ensure_ascii=False) + "\n")
                self.cache[k] = rec
                return rec
            except Exception as e:  # noqa: BLE001
                err = e
        return {"key": k, "model": self.model, "out": {"error": str(err)}, "in_tok": 0, "out_tok": 0}


llm = LLMExtractor(MODEL, f"{OUT_DIR}/llm_cache.jsonl")

## 5. Build the table (one row per prompt)

In [ ]:
ATTRS = ["category", "label", "intensity", "severity", "risk_type", "negated", "experiencer",
         "temporality", "hypothetical", "certainty", "mi_talk", "mi_talk_evidence"]


def cell(v):
    return "; ".join(v) if isinstance(v, list) else v


def build_table(name, sel, recs):
    rows = []
    for (_, r), rec in zip(sel.iterrows(), recs):
        o = rec["out"]
        tl = norm(r.prompt).lower()
        spans = [] if "error" in o else o.get("spans", [])
        pos = lambda s: tl.find(norm(s).lower())
        spans = sorted(spans, key=lambda s: (pos(s) < 0, pos(s)))        # order of appearance in the prompt
        rows.append({
            "dataset": name, "utt_id": r.utt_id, "transcript_id": r.transcript_id,
            "prompt": r.prompt,
            "spans": " | ".join(spans),
            **{a: cell(o.get(a, "")) for a in ATTRS},
            "core_meaning": o.get("core_meaning", ""),
            "n_spans": len(spans),
            "prompt_char_len": r.prompt_char_len, "prompt_word_len": r.prompt_word_len,
            "nonverbatim_spans": " | ".join(s for s in spans if pos(s) < 0),   # check these by hand
            "llm_error": o.get("error", ""),
        })
    return pd.DataFrame(rows)


def run(rows, sys_prompt=SYS):
    recs = [llm.extract(t, sys_prompt) for t in tqdm(rows["prompt"], desc="extracting")]
    print(f"in/out tokens: {sum(r['in_tok'] for r in recs)}/{sum(r['out_tok'] for r in recs)}")
    return recs

### 6a. Test on 5 prompts per dataset first — check the output looks right

In [ ]:
for name, sel in sels.items():
    test = sel.head(5)
    print(name); display(build_table(name, test, run(test, sys_for(DATASETS[name]))))

### 6b. Full run (20 prompts per dataset; reuses the cache from 6a)

In [ ]:
results = {}
for name, sel in sels.items():
    table = build_table(name, sel, run(sel, sys_for(DATASETS[name])))
    results[name] = table
    print(f"{name}: prompts {len(table)} | spans {table.n_spans.sum()} | "
          f"prompts with non-verbatim spans {(table.nonverbatim_spans != '').sum()} | LLM errors {(table.llm_error != '').sum()}")

## 7. Save

In [ ]:
for name, table in results.items():
    path = f"{OUT_DIR}/{name}_mustpreserve.csv"
    table.to_csv(path, index=False)
    print("saved", path, table.shape)

try:  # Colab: download the CSVs
    from google.colab import files as colab_files
    for name in results:
        colab_files.download(f"{OUT_DIR}/{name}_mustpreserve.csv")
except Exception:
    pass